# CIFAR-10 Kaggle Competition — Custom ResNet (PyTorch)

Trains a small CIFAR-style ResNet from scratch (random init, no pretrained weights) on the official CIFAR-10 training set, and generates predictions for the Kaggle competition's 300k-image test set.

In [2]:
import os
import time
import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from PIL import Image

import sys
sys.path.append('src')
from model import ResNetCIFAR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

device: cuda
NVIDIA GeForce RTX 3060 Laptop GPU


## 1. Data augmentation (★ your turn)

CIFAR-10's official train/test split is 50,000 / 10,000 images, each 32x32 RGB.
Without augmentation, a ResNet-20 will comfortably hit ~90%+ *training* accuracy
but overfit — validation accuracy plateaus lower, typically 0.83-0.86 on CIFAR-10.

The classic CIFAR-10 recipe (and the single biggest lever for pushing past 0.88)
is:
- `RandomCrop(32, padding=4)` — pad the image by 4px on each side then crop back
  to 32x32, so the network sees the object shifted around instead of always
  centered.
- `RandomHorizontalFlip()` — cheap, valid for natural images (a horizontally
  flipped cat is still a cat), and does *not* make sense for e.g. digit
  classification (a flipped "5" isn't a "5").
- Normalize with CIFAR-10's per-channel mean/std (commonly `(0.4914, 0.4822,
  0.4465)` / `(0.2470, 0.2435, 0.2616)`).

Optionally, `transforms.RandomErasing()` (a form of Cutout) after `ToTensor()`
randomly blacks out a rectangular patch — forces the network to not rely on
any single localized feature.

**Fill in `train_transform` below.** The validation/test transform should
*only* resize/normalize — no randomness, since we want deterministic
evaluation.

In [3]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Deterministic transform for validation/test/Kaggle inference: just
# ToTensor() + Normalize, no randomness.
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

assert train_transform is not None, "Fill in train_transform above"

## 2. Datasets

- **Training data**: the *official* CIFAR-10 train/test split via
  `torchvision.datasets.CIFAR10(download=True)`, per the assignment ("you have
  the option to train your model on the official CIFAR10 data"). We hold out
  the official CIFAR-10 test set (10,000 labeled images) purely as our own
  validation/estimate of what the Kaggle score should be — we never train on it.
- **Kaggle test set**: `data/test/` contains 300,000 unlabeled PNGs named
  `1.png` ... `300000.png`, extracted from `test.7z`. We need a custom
  `Dataset` that reads them in `id` order so predictions line up with
  `sampleSubmission.csv`.

In [4]:
train_full = torchvision.datasets.CIFAR10(
    root='data/torchvision', train=True, download=True, transform=train_transform
)
official_test = torchvision.datasets.CIFAR10(
    root='data/torchvision', train=False, download=True, transform=eval_transform
)

CLASSES = train_full.classes  # ['airplane', 'automobile', 'bird', ..., 'truck'] — alphabetical
print(CLASSES)
print('train_full:', len(train_full), 'official_test:', len(official_test))

['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
train_full: 50000 official_test: 10000


In [5]:
class KaggleTestDataset(Dataset):
    """Reads data/test/<id>.png for id in 1..n, in ascending numeric id order."""

    def __init__(self, root: str, transform=None):
        paths = glob.glob(os.path.join(root, '*.png'))
        # sort numerically by filename (1.png, 2.png, ..., 300000.png), not lexicographically
        self.paths = sorted(paths, key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
        self.ids = [int(os.path.splitext(os.path.basename(p))[0]) for p in self.paths]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, self.ids[idx]

In [6]:
BATCH_SIZE = 128

train_loader = DataLoader(train_full, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(official_test, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

kaggle_test = KaggleTestDataset('data/test', transform=eval_transform)
# num_workers=0 on purpose: multi-worker DataLoaders here have repeatedly
# hung/deadlocked on this machine (Windows + Jupyter + an already-initialized
# CUDA context is a known bad combination for spawning DataLoader workers).
# Single-threaded decoding is slower per-core but actually completes.
test_loader = DataLoader(kaggle_test, batch_size=1024, shuffle=False, num_workers=0, pin_memory=True)

print('Kaggle test images found:', len(kaggle_test))
assert len(kaggle_test) == 300000, f"expected 300000 test images, found {len(kaggle_test)}"

Kaggle test images found: 300000


























## 3. Model (★ your turn — see `src/model.py`)

Open `src/model.py` and fill in `BasicBlock.__init__` (the `self.shortcut`)
and `BasicBlock.forward`. The docstring in that file walks through what a
residual block does and why. `ResNetCIFAR` (already written) wires 3 stages
of `BasicBlock`s together into a full network — `num_blocks_per_stage=3`
gives ResNet-20 (20 = 6*3 + 2), a good fit for a 6GB GPU.

Run the cell below once you've implemented it — it does a forward pass on a
dummy batch as a smoke test before we commit to a full training run.

In [7]:
# reload src.model in case you edited it after the notebook started
import importlib
import model as model_module
importlib.reload(model_module)
from model import ResNetCIFAR

net = ResNetCIFAR(num_blocks_per_stage=3, num_classes=10).to(device)
n_params = sum(p.numel() for p in net.parameters())
print(f'{n_params:,} parameters')


# smoke test: dummy batch, confirm output shape and that weights are randomly initialized
dummy = torch.randn(4, 3, 32, 32, device=device)
out = net(dummy)
assert out.shape == (4, 10), out.shape
print('forward pass OK, output shape', out.shape)

# quick random-init sanity check: two freshly constructed nets should NOT have identical weights
net2 = ResNetCIFAR(num_blocks_per_stage=3, num_classes=10)
same = torch.equal(net.stem[0].weight.detach().cpu(), net2.stem[0].weight.detach())
print('two fresh nets have identical stem weights (should be False):', same)

272,474 parameters
forward pass OK, output shape torch.Size([4, 10])
two fresh nets have identical stem weights (should be False): False


## 4. Training setup (★ your turn — optimizer + LR schedule)

Two well-known choices for CIFAR-10 with a ResNet:

- **SGD + momentum + cosine annealing** (the recipe from the original ResNet
  paper, updated with cosine decay instead of step decay). Typically:
  `optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)`
  with `optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)`.
  Needs more epochs to converge but tends to generalize best — this is the
  most reliable path to 0.90+ on CIFAR-10.
- **AdamW + OneCycleLR**: `optim.AdamW(net.parameters(), lr=3e-3, weight_decay=5e-4)`
  with `optim.lr_scheduler.OneCycleLR(optimizer, max_lr=..., steps_per_epoch=len(train_loader), epochs=EPOCHS)`.
  Converges faster (fewer epochs needed) and is more forgiving of the exact
  LR you pick, but can plateau slightly lower than a well-tuned SGD run.

Either is a fine choice here (SGD+cosine is the safer bet for max accuracy in
a fixed epoch budget). **Fill in `optimizer` and `scheduler` below**, along
with `EPOCHS` (~40-60 is a reasonable budget on a 3060 for ResNet-20) and
`criterion` (cross-entropy for a 10-class problem — one line).

In [8]:
import torch.optim as optim

EPOCHS = 50
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    net.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

assert None not in (EPOCHS, criterion, optimizer, scheduler), "Fill in EPOCHS / criterion / optimizer / scheduler above"

## 5. Training loop

Uses `torch.amp` mixed precision (fp16 compute, fp32 master weights) since the
3060 only has 6GB — this roughly halves memory use and speeds up training
with negligible accuracy impact. Validates each epoch on the *official*
CIFAR-10 test set and checkpoints the best model by validation accuracy to
`best_model.pt`. Per the assignment, this validation accuracy is our estimate
of the Kaggle score before we ever touch the Kaggle test set.

In [9]:
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))


def run_epoch(loader, train: bool):
    net.train(train)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, targets in loader:
            images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                outputs = net(images)
                loss = criterion(outputs, targets)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += images.size(0)

    return total_loss / total, correct / total


# Resume-safe: if a checkpoint already exists on disk, find out how good it
# actually is before training starts, so a restart/rerun can never silently
# overwrite better weights with worse ones (this bit us once already).
best_val_acc = 0.0
if os.path.exists('best_model.pt'):
    net.load_state_dict(torch.load('best_model.pt', map_location=device))
    _, best_val_acc = run_epoch(val_loader, train=False)
    print(f'Found existing best_model.pt with val_acc {best_val_acc:.4f} -- will only overwrite if beaten.')

history = []

for epoch in range(EPOCHS):
    t0 = time.time()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(net.state_dict(), 'best_model.pt')

    history.append((epoch, train_loss, train_acc, val_loss, val_acc))
    print(f'epoch {epoch+1:>3}/{EPOCHS}  '
          f'train_loss {train_loss:.4f} train_acc {train_acc:.4f}  '
          f'val_loss {val_loss:.4f} val_acc {val_acc:.4f}  '
          f'best {best_val_acc:.4f}  ({time.time()-t0:.1f}s)')

print(f'\nBest validation accuracy (estimate of Kaggle score): {best_val_acc:.4f}')


Found existing best_model.pt with val_acc 0.4810 -- will only overwrite if beaten.
epoch   1/50  train_loss 1.0218 train_acc 0.6373  val_loss 1.0713 val_acc 0.6260  best 0.6260  (48.1s)
epoch   2/50  train_loss 0.8099 train_acc 0.7165  val_loss 0.8681 val_acc 0.6998  best 0.6998  (44.3s)
epoch   3/50  train_loss 0.7119 train_acc 0.7516  val_loss 0.8809 val_acc 0.7010  best 0.7010  (49.1s)
epoch   4/50  train_loss 0.6524 train_acc 0.7752  val_loss 0.9135 val_acc 0.6957  best 0.7010  (46.9s)
epoch   5/50  train_loss 0.6199 train_acc 0.7870  val_loss 0.9156 val_acc 0.7031  best 0.7031  (46.7s)
epoch   6/50  train_loss 0.5896 train_acc 0.7958  val_loss 0.8624 val_acc 0.7056  best 0.7056  (42.9s)
epoch   7/50  train_loss 0.5716 train_acc 0.8046  val_loss 0.7341 val_acc 0.7511  best 0.7511  (41.8s)
epoch   8/50  train_loss 0.5511 train_acc 0.8110  val_loss 0.6608 val_acc 0.7785  best 0.7785  (42.1s)
epoch   9/50  train_loss 0.5425 train_acc 0.8138  val_loss 0.9092 val_acc 0.7044  best 0.7785

## 6. Predict on the Kaggle test set and write `submission.csv`

Loads the best checkpoint (by validation accuracy), runs inference over all
300,000 Kaggle test images, and writes predictions in the `id,label` format
`sampleSubmission.csv` expects (label as the class name string).

In [10]:
net.load_state_dict(torch.load('best_model.pt', map_location=device))
net.eval()

all_ids, all_preds = [], []
t0 = time.time()
with torch.inference_mode():
    for batch_idx, (images, ids) in enumerate(test_loader):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = net(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_ids.extend(ids.numpy().tolist())
        all_preds.extend(preds.tolist())

        if batch_idx % 20 == 0:
            print(f'  batch {batch_idx}: {len(all_ids):,} images done ({time.time()-t0:.1f}s elapsed)')

elapsed = time.time() - t0
print(f'inference over {len(all_ids):,} images in {elapsed:.1f}s ({len(all_ids)/elapsed:.0f} images/s)')

submission = pd.DataFrame({
    'id': all_ids,
    'label': [CLASSES[p] for p in all_preds],
})
submission = submission.sort_values('id').reset_index(drop=True)

assert len(submission) == 300000, len(submission)
assert set(submission['label']) <= set(CLASSES)

submission.to_csv('submission.csv', index=False)
submission.head()

  batch 0: 1,024 images done (1.9s elapsed)
  batch 20: 21,504 images done (28.0s elapsed)
  batch 40: 41,984 images done (53.6s elapsed)
  batch 60: 62,464 images done (76.7s elapsed)
  batch 80: 82,944 images done (99.2s elapsed)
  batch 100: 103,424 images done (121.0s elapsed)
  batch 120: 123,904 images done (142.9s elapsed)
  batch 140: 144,384 images done (164.6s elapsed)
  batch 160: 164,864 images done (186.3s elapsed)
  batch 180: 185,344 images done (207.7s elapsed)
  batch 200: 205,824 images done (229.4s elapsed)
  batch 220: 226,304 images done (251.3s elapsed)
  batch 240: 246,784 images done (273.3s elapsed)
  batch 260: 267,264 images done (295.0s elapsed)
  batch 280: 287,744 images done (318.0s elapsed)
inference over 300,000 images in 331.4s (905 images/s)


,id,label
0,1,deer
1,2,deer
2,3,automobile
3,4,ship
4,5,bird


## 7. Submit to Kaggle

Run from a terminal (not this cell) once `submission.csv` looks right:

```
kaggle competitions submit -c cifar-10 -f submission.csv -m "ResNet-20, SGD+cosine, N epochs"
```

Then check your score at https://www.kaggle.com/c/cifar-10/leaderboard (it will
show as a **late submission** since the competition is closed) and screenshot
it for the report.

## Notes for the report

- **Architecture**: ResNet-20 for CIFAR (3 stages x 3 BasicBlocks, 16→32→64
  channels, ~0.27M params), implemented from scratch in `src/model.py`.
- **Weights**: randomly initialized (PyTorch default init) — no
  `pretrained=True`, no loaded `state_dict` anywhere in the pipeline.
- **Data**: official CIFAR-10 train set (50,000 images) via
  `torchvision.datasets.CIFAR10`, with `<describe your train_transform choice
  here>` augmentation.
- **Training**: `<describe your optimizer/scheduler/EPOCHS choice here>`,
  mixed precision, best checkpoint selected by accuracy on the official
  CIFAR-10 test set (10,000 images) — final validation accuracy: `<fill in
  best_val_acc>`.
- **Kaggle score**: `<fill in after submitting>`.